# Notebook 02 — Fontes de Dados do Projeto

Este notebook descreve todas as fontes de dados utilizadas no projeto, a estratégia de coleta, os cuidados com padronização e os critérios para evitar vazamento de informação entre os conjuntos de treino, validação e teste.


## 1. Visão Geral das Fontes

O projeto utiliza duas fontes de dados complementares:

| Fonte | Cobertura temporal | Tipo de dado | Papel no projeto |
|---|---|---|---|
| **CSV históricos** (GitHub / Kaggle) | 2019–2025 | Resultados finais por corrida | Pipeline 1 e Pipeline 2 |
| **API OpenF1** | 2023–presente | Dados de sessão, telemetria, clima, eventos | Pipeline 3 |

Os CSVs históricos são suficientes para os Pipelines 1 e 2, que trabalham apenas com o ranking final de cada corrida. A OpenF1 é incorporada no Pipeline 3 para enriquecer o modelo com contexto intra-corrida.


## 2. Fonte 1 — Dados Históricos em CSV

### 2.1. Origem

Os dados em CSV provêm de duas coleções públicas:

- **GitHub — toUpperCase78/formula1-datasets**
  `https://github.com/toUpperCase78/formula1-datasets`
  Estrutura organizada por temporada, cobertura de 2019 em diante, fácil integração.

- **Kaggle — Formula 1 World Championship 1950–2020**
  `https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020`
  Cobertura histórica ampla, mais adequada para análises de longo prazo.

Para o escopo atual do projeto, utiliza-se principalmente a coleção do GitHub.

---

### 2.2. Organização no repositório

```
data/
├── Season2019/
├── Season2020/
├── Season2021/
├── Season2022/
├── Season2023/
├── Season2024/
└── Season2025/
```

Cada pasta de temporada contém arquivos CSV com nomenclatura padronizada:

| Arquivo | Conteúdo | Uso no projeto |
|---|---|---|
| `raceResults.csv` | Resultado final da corrida: posição, piloto, equipe, pontos, status | **Principal** — Rankings parciais top-k |
| `qualifyingResults.csv` | Resultados da qualificação (Q1, Q2, Q3) | Disponível para features futuras |
| `SprintResults.csv` | Resultado das corridas sprint | Separado do resultado principal |
| `SprintQualifyingResults.csv` | Resultado da sprint qualifying | Disponível para features futuras |
| `drivers.csv` | Cadastro dos pilotos: nome, sigla, número, nacionalidade | Padronização de nomes |
| `calendar.csv` | Calendário: etapa, data, circuito, país | Sequência cronológica das corridas |

---

### 2.3. Campos relevantes do `raceResults.csv`

O arquivo de resultado de corrida é a entrada principal do pipeline. Os campos utilizados são:

| Campo | Tipo | Descrição |
|---|---|---|
| `Position` | int / string | Posição final; valores não numéricos indicam DNF/DNS/DSQ |
| `Driver` | string | Nome completo do piloto |
| `Team` | string | Equipe |
| `Points` | float | Pontos marcados |
| `Race` | string | Nome da etapa (usado para identificar o circuito) |
| `Season` | int | Temporada |

A coluna `Position` é normalizada: valores numéricos são convertidos para `int`, e valores como `DNF`, `DNS`, `DSQ`, `NC` são marcados como abandono.

---

### 2.4. Padronização de nomes de pilotos

Diferentes fontes de dados escrevem nomes de pilotos de formas distintas (ex: `"Zhou Guanyu"` vs `"Guanyu Zhou"`). O módulo `src/data/data_pipeline.py` mantém um dicionário `DRIVER_ABBREV` que converte nomes completos em siglas de três letras padronizadas pelo próprio campeonato:

```python
"Max Verstappen"    → "VER"
"Lewis Hamilton"    → "HAM"
"Charles Leclerc"   → "LEC"
"Lando Norris"      → "NOR"
"Zhou Guanyu"       → "ZHO"
"Guanyu Zhou"       → "ZHO"
```

Essa padronização é essencial para integrar dados de múltiplas fontes e garantir consistência entre temporadas.

---

### 2.5. Limitações dos CSVs históricos

- Contêm apenas o **resultado final** da corrida, sem informação do que ocorreu durante a prova.
- Não diferenciam as **causas de abandono**: falha mecânica, acidente, colisão, estratégia.
- Não incluem dados de **clima, safety car, pit stops ou pneus**.
- A qualificação está disponível apenas a partir de 2022 em alguns datasets.
- A estrutura de colunas pode variar levemente entre temporadas, exigindo normalização.


## 3. Fonte 2 — API OpenF1

### 3.1. O que é a OpenF1

A **OpenF1** é uma API aberta e gratuita para dados de Fórmula 1, desenvolvida e mantida pela comunidade.

- **Site:** https://openf1.org/
- **Repositório:** https://github.com/br-g/openf1
- **Formato de resposta:** JSON (conversível para CSV ou Parquet)
- **Cobertura:** dados históricos completos a partir de 2023; dados parciais disponíveis para temporadas anteriores em alguns endpoints.
- **Autenticação:** não requerida para uso público.

---

### 3.2. Por que incorporar a OpenF1 ao projeto

Os CSVs históricos permitem construir modelos baseados no **ranking final** de cada corrida. A OpenF1 permite ir além, fornecendo dados sobre o **processo** que gerou esse resultado:

1. **Contexto climático.** Temperatura de pista e ar, umidade e presença de chuva afetam estratégias, degradação de pneus e probabilidade de acidentes. Esses dados permitem construir features contextualmente informativas.

2. **Eventos de controle de corrida.** Safety cars, bandeiras amarelas e vermelhas alteram profundamente o resultado final. Um modelo que ignora esses eventos trata corridas com safety car como equivalentes a corridas limpas.

3. **Informações sobre abandono.** A OpenF1 permite identificar em que volta um piloto abandonou e qual foi a causa registrada, tornando possível modelar a probabilidade de DNF como função do circuito, piloto e equipe.

4. **Grid de largada.** A posição de largada é um preditor muito informativo da posição de chegada. Com o endpoint `starting_grid`, esse dado fica disponível como feature pré-corrida.

5. **Dados de stints e pit stops.** Estratégias de pneus influenciam fortemente o resultado, especialmente em corridas com alta degradação.

---

### 3.3. Endpoints relevantes para o projeto

| Endpoint | Dados fornecidos | Uso planejado |
|---|---|---|
| `meetings` | Grandes Prêmios: data, país, circuito | Identificação e indexação das corridas |
| `sessions` | Sessões: tipo (corrida, quali, sprint), chave | Navegação entre sessões |
| `drivers` | Pilotos: sigla, número, nome, equipe | Padronização de identificadores |
| `starting_grid` | Grade de largada da corrida | Feature pré-corrida: posição de largada |
| `session_result` | Resultado oficial: posição, status, pontos | Alternativa ou complemento ao CSV |
| `laps` | Tempo por volta, setores, velocidade | Análise de ritmo e consistência |
| `stints` | Composto de pneu, número de voltas por stint | Features de estratégia |
| `pit` | Pit stops: volta de entrada, duração | Features de estratégia |
| `position` | Evolução de posição ao longo da corrida | Análise dinâmica do resultado |
| `intervals` | Gap para líder e carro à frente | Análise de ritmo relativo |
| `race_control` | Safety car, bandeiras, penalizações | Identificação de eventos aleatórios |
| `weather` | Temperatura (ar e pista), umidade, chuva, vento | Features climáticas por corrida |
| `car_data` | Velocidade, acelerador, freio, marcha, RPM, DRS | Telemetria (análise exploratória) |
| `championship_drivers` | Classificação do campeonato de pilotos | Feature de contexto competitivo |
| `championship_teams` | Classificação do campeonato de construtores | Feature de contexto competitivo |

---

### 3.4. Estratégia de coleta e cache local

Chamar a API repetidamente a cada execução do pipeline é ineficiente e desnecessário. A estratégia adotada é:

1. **Coleta única por temporada/corrida:** os dados são requisitados uma vez e salvos localmente.
2. **Cache em `data/openf1/`:** estrutura de diretórios separada por tipo de dado e temporada.
3. **Formato de armazenamento:** CSV para tabelas planas; Parquet para dados de telemetria (maior eficiência de leitura).

```
data/openf1/
├── raw/
│   ├── meetings_2023.csv
│   ├── sessions_2023.csv
│   ├── weather_2023_bahrain_race.csv
│   ├── race_control_2023_bahrain_race.csv
│   └── ...
└── processed/
    ├── race_context_2023.csv      # Features por corrida (clima, safety car, grid)
    ├── race_context_2024.csv
    └── race_context_2025.csv
```

O módulo `src/pipeline_openf1/openf1_client.py` centraliza as funções de acesso à API, e `openf1_cache.py` gerencia a persistência local, verificando se os dados já foram baixados antes de fazer novas requisições.

---

### 3.5. Cuidados com vazamento de informação

Este ponto é crítico. Ao incorporar dados da OpenF1, é necessário classificar cada feature em uma de três categorias:

| Categoria | Exemplos | Pode ser usada para prever? |
|---|---|---|
| **Pré-corrida** | Grid de largada, previsão do tempo, classificação do campeonato | ✅ Sim — disponível antes da largada |
| **Durante a corrida** | Posições em pista, pit stops, safety car em andamento | ❌ Não — só disponível em cenários in-race |
| **Pós-corrida** | Resultado final, tempos de volta, resultado de pit stops | ❌ Não para a corrida atual; ✅ para corridas passadas |

## 4. Comparação entre as Fontes

| Critério | CSVs históricos | API OpenF1 |
|---|---|---|
| **Cobertura temporal** | 2019–2025 | 2023–presente (completo) |
| **Tipo de dado** | Resultado final | Processo completo da corrida |
| **Facilidade de uso** | Alta — arquivos locais | Média — requer coleta e cache |
| **Riqueza** | Baixa — apenas resultado | Alta — clima, eventos, telemetria |
| **Cuidado com vazamento** | Baixo — apenas resultado | Alto — dados durante a corrida |
